In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import glasbey
import seaborn as sns
import anndata as ad
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import umap
from sklearn.cluster import KMeans
from pycombat import Combat  # or from neuroCombat import neuroCombat
import itertools

dir = '/Users/jeinhaus/Documents/03 Stanford data/OSCC Recurrence/PythonIMC'
sys.path.append(os.path.abspath(dir))
from pythonimc import assembling, plotting

base_dir = '/Users/jeinhaus/Library/CloudStorage/GoogleDrive-jeinhaus@stanford.edu/Shared drives/Oral Immunology Projects/OSCC Recurrence/Clean_analysis'

/Users/jeinhaus/miniconda3/envs/OSCC/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'pycombat'

Phenotyping

In [2]:
data = ad.read_h5ad('/Users/jeinhaus/Documents/03 Stanford data/OSCC Recurrence/12012025_annotdata.h5ad')

data.obs['Casenumber'] = data.obs['Casenumber'].str.upper()

data.obs["TMA"] = data.obs["Image"].str[0]

# Exclude Background and TMA controls
mask = (data.obs['Zone'] != 'Background') & (~data.obs['Casenumber'].str.contains('TONSIL|LIVER', na=False))
data = data[mask].copy()

data.obs['Region'] = data.obs['Zone'].str.split("_").str[0]
data.obs['Sector'] = data.obs['Zone'].str.split("_").str[1]

# Add panel
panel = pd.read_csv(os.path.join(base_dir, "panel.csv"))
data.var['marker'] = panel['marker'].tolist()
data.var['channel_marker'] = panel['name'].tolist()
data.var.index = data.var['marker']

# Metadata
#data = assembling.add_metadata(data, base_dir=base_dir, metadata_files=['Outcomes_Stanford.xlsx','Outcomes_Tubingen.xlsx'], merge_on_column="Casenumber")

gc.collect()

42722

# Densities

In [13]:
counts = (
    data.obs
    .groupby(["Image", "Sector", "Subtype"])
    .size()
    .reset_index(name="Count")
)

# Extract unique image → area mapping from obs
areas = data.obs[["Image", "Casenumber", "Tumorcore", "Tumorborder", "Stromacore", "Stromaborder"]].drop_duplicates()
areas = areas.groupby("Image").sum().reset_index()

# Merge counts with area info
merged = counts.merge(areas, on="Image", how="left")

# Compute density per zone using the correct area column dynamically
zone_to_area = {
    "Tumorcore": "Tumorcore",
    "Tumorborder": "Tumorborder",
    "Stromacore": "Stromacore",
    "Stromaborder": "Stromaborder"
}

merged["Area"] = merged.apply(lambda x: x[zone_to_area.get(x["Sector"], None)], axis=1)
merged["Density"] = np.where(
    merged["Area"] > 0,
    (merged["Count"] / merged["Area"]) * 1_000_000,
    np.nan  # put NaN if Area == 0
)

# Pivot to wide format: columns = Subtype–Sector
density_df = merged.pivot_table(
    index="Image",
    columns=["Subtype", "Sector"],
    values="Density",
    fill_value=0
)

# Optional: flatten column MultiIndex
density_df.columns = [f"{subtype}_{sector}_density" for subtype, sector in density_df.columns]
densities_wide = density_df

/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/3831675254.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Image", "Sector", "Subtype"])
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/3831675254.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  areas = areas.groupby("Image").sum().reset_index()
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/3831675254.py:31: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain 

# Frequencies

In [24]:
# Count cells per Image, Sector, Subtype
counts = (
    data.obs
    .groupby(["Image", "Sector", "Subtype"])
    .size()
    .reset_index(name="Count")
)

# Compute total cells per image *and zone*
total_per_zone = (
    counts
    .query('Subtype != "Other"')  # 👈 exclude unclassified cells
    .groupby(["Image", "Sector"])["Count"]
    .sum()
    .reset_index(name="Total_excl_Other")
)

# Merge totals (only affects the denominator)
freq_df = counts.merge(total_per_zone, on=["Image", "Sector"], how="left")

# Compute frequency only relative to classified cells
freq_df["Frequency"] = freq_df["Count"] / freq_df["Total_excl_Other"] * 100

# Pivot to wide table
freq_wide = freq_df.pivot_table(
    index="Image",
    columns=["Subtype", "Sector"],
    values="Frequency",
    fill_value=0
)

# Flatten MultiIndex columns
freq_wide.columns = [f"{subtype}_{sector}_freq" for subtype, sector in freq_wide.columns]
#freq_wide = freq_wide.loc[:, ~freq_wide.columns.str.replace('_freq$', '', regex=True).isin(zero_median_cols)]
print(freq_wide.shape)

/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_89078/3216945674.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Image", "Sector", "Subtype"])


(572, 112)


/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_89078/3216945674.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Image", "Sector"])["Count"]
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_89078/3216945674.py:25: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  freq_wide = freq_df.pivot_table(


# Ratios

In [ ]:
counts_df = (
    data.obs
    .groupby(["Image", "Sector", "Subtype"])
    .size()
    .reset_index(name="Count")
)

counts_wide = counts_df.pivot_table(
    index=["Image", "Sector"],
    columns="Subtype",
    values="Count",
    fill_value=0
)

ratio_df = counts_wide.copy()

subtypes = counts_wide.columns.tolist()

# Generate unique unordered pairs
pairs = list(itertools.combinations(subtypes, 2))

for a, b in pairs:
    # Add small epsilon to avoid division by zero
    ratio_df[f"{a}_to_{b}_ratio"] = (counts_wide[a] + 1) / (counts_wide[b] + 1)

ratios_wide = (
    ratio_df
    .pivot_table(index="Image", columns="Sector")
)

# Flatten multiindex columns
ratios_wide.columns = [f"{sector}_{col}" for col, sector in ratios_wide.columns]
cols_to_keep = [col for col in ratios_wide.columns if 'ratio' in col.lower()]
ratios_wide = ratios_wide[cols_to_keep]

# Split zero_median_cols into tuples of two parts
#zero_parts = [tuple(name.split('_', 1)) for name in zero_median_cols]

# # Function to check if a column contains both parts of any zero_median feature
# def should_keep(col):
#     col_parts = col.split('_')
#     return not any(part1 in col_parts and part2 in col_parts for part1, part2 in zero_parts)

# Apply mask
#ratios_wide = ratios_wide.loc[:, [should_keep(c) for c in ratios_wide.columns]]
print(ratios_wide.shape)

# Sectorratios

In [ ]:
counts_df = (
    data.obs
    .groupby(["Image", "Sector", "Subtype"])
    .size()
    .reset_index(name="Count")
)

counts_wide = counts_df.pivot_table(
    index=["Image", "Sector"],
    columns="Subtype",
    values="Count",
    fill_value=0
)

# Pivot to: index = Image, columns = Sector_Subtype
sector_wide = counts_df.pivot_table(
    index="Image",
    columns=["Sector", "Subtype"],
    values="Count",
    fill_value=0
)
sector_wide.columns = [f"{sector}_{subtype}" for sector, subtype in sector_wide.columns]

sector_ratio_df = sector_wide.copy()

subtypes = counts_df["Subtype"].unique()
sectors = counts_df["Sector"].unique()

sector_pairs = list(itertools.combinations(sectors, 2))

for subtype in subtypes:
    for s1, s2 in sector_pairs:
        col1 = f"{s1}_{subtype}"
        col2 = f"{s2}_{subtype}"
        if col1 in sector_wide.columns and col2 in sector_wide.columns:
            sector_ratio_df[f"{subtype}_{s1}_to_{s2}_ratio"] = \
                (sector_wide[col1] + 1) / (sector_wide[col2] + 1)
            
sector_ratio_df = sector_ratio_df[[c for c in sector_ratio_df.columns if "ratio" in c]]

# Areas

In [35]:
# 1️⃣ Sum cell areas per Image–Subtype–Sector
subtype_area = (
    data.obs
    .groupby(["Image", "Sector"], as_index=False)["Area"]
    .sum()
    .rename(columns={"Area": "SumCellArea"})
    .pivot_table(index="Image", columns="Sector", values="SumCellArea", fill_value=0)
    .rename(columns=lambda x: f"{x}_sumcellarea")
)

# 2️⃣ Extract total sector area per Image–Sector
sector_totals = (
    data.obs.groupby(["Image"], as_index=False)
    .first()  # first row in each group
    [["Image", "Tumorcore", "Tumorborder", "Stromaborder", "Stromacore"]]  # sector area columns
)

areas_wide = subtype_area.merge(sector_totals, on="Image")
areas_wide["Stromaborder_Tumorborder_ratio"] = areas_wide["Stromaborder"] / areas_wide["Tumorborder"]

areas_wide["Stromaborder_Tumorcore_ratio"] = np.where(
    areas_wide["Tumorcore"] != 0,
    areas_wide["Stromaborder"] / areas_wide["Tumorcore"],
    np.nan
)
areas_wide["Tumorborder_Tumorcore_ratio"] = np.where(
    areas_wide["Tumorcore"] != 0,
    areas_wide["Tumorborder"] / areas_wide["Tumorcore"],
    np.nan
)

areas_wide["Tumorcore_cellular"] = areas_wide["Tumorcore_sumcellarea"] / areas_wide["Tumorcore"]
areas_wide["Tumorborder_cellular"] = areas_wide["Tumorborder_sumcellarea"] / areas_wide["Tumorborder"]
areas_wide["Stromaborder_cellular"] = areas_wide["Stromaborder_sumcellarea"] / areas_wide["Stromaborder"]
areas_wide["Stromacore_cellular"] = areas_wide["Stromacore_sumcellarea"] / areas_wide["Stromacore"]

areas_wide["Tumorcore_acellular"] = (areas_wide["Tumorcore_cellular"] - 1).abs()
areas_wide["Tumorborder_acellular"] =  (areas_wide["Tumorborder_cellular"] - 1).abs()
areas_wide["Stromaborder_acellular"] =  (areas_wide["Stromaborder_cellular"] - 1).abs()
areas_wide["Stromacore_acellular"] = (areas_wide["Stromacore_cellular"] - 1).abs()

# Count datapoints per (Image, Sector, Subtype)
grouped = (
    data.obs
    .groupby(["Image", "Sector", "Subtype"])
    .agg(n=("Area", "size"), mean_area=("Area", "mean"))
    .reset_index()
)

# Keep only groups with >10 datapoints
filtered = grouped[grouped["n"] > 10]


filtered["Feature"] = filtered["Subtype"].astype(str) + "_" + filtered["Sector"].astype(str) + "_area"
cellarea_wide = filtered.pivot_table(
    index="Image",
    columns=["Feature"],
    values="mean_area"
)

areas_wide = areas_wide.merge(cellarea_wide, on="Image")

# areas_wide = areas_wide.loc[
#     :, 
#     ~areas_wide.columns.str.rsplit('_', n=1).str[0].isin(zero_median_cols)
# ]
areas_wide = areas_wide.set_index('Image')

/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/4150446420.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data.obs
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/4150446420.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  data.obs
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/4150446420.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data.obs.groupby(["Image"], as_index=False)
/var/fol

# Functional

In [54]:
# Define the markers of interest
functional_markers = ["Ki67", "SOX9", "pMK2", "pS6", "COX2", "HIFa", "NFkB", "STAT3", "CD36",
                      "GRZB", "NE", "MMP9", "MPO", "PDL1", "PD1", "Vista", "ICOS", "VEGF", "CD44", "HLADR", "TenC"]

# Make sure all markers exist in your data
markers = [m for m in data.var_names]# m in functional_markers if m in data.var_names]

# Get the arcsinh-transformed counts (or raw counts if you prefer)
X = pd.DataFrame(data.layers['corrected'],
                 columns=data.var_names,
                 index=data.obs_names)

# Add obs information: Image, Subtype, Sector (you called it 'Sector', adjust if different)
X = X.join(data.obs[['Image', 'Subtype', 'Sector']])

# First, compute counts per group
counts = (
    X.groupby(["Image", "Subtype", "Sector"])
    .size()
)

# Select only groups with >10
valid_groups = counts[counts > 10].index

# Group by Image, Subtype, Sector and take median
median_df = X.groupby(['Image', 'Subtype', 'Sector'])[markers].mean().loc[valid_groups].reset_index()

# Melt the marker columns to long format
median_long = median_df.melt(
    id_vars=['Image', 'Subtype', 'Sector'],
    value_vars=functional_markers,
    var_name='Marker',
    value_name='Expression'
)

# Create combined feature name: Subtype_Marker_Sector
median_long['Feature'] = median_long['Subtype'].astype(str) + "_" + median_long['Sector'].astype(str) + "_" + median_long['Marker'].astype(str)

# Pivot so that Images are rows and Features are columns
median_wide = median_long.pivot_table(
    index='Image',
    columns='Feature',
    values='Expression'
)

# median_wide = median_wide.loc[
#     :, 
#     ~median_wide.columns.str.rsplit('_', n=1).str[0].isin(zero_median_cols)
# ]
print(median_wide.shape)

/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/730956052.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  X.groupby(["Image", "Subtype", "Sector"])
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/730956052.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  median_df = X.groupby(['Image', 'Subtype', 'Sector'])[markers].mean().loc[valid_groups].reset_index()
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/730956052.py:40: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify 

(572, 1869)


# Clinical

In [ ]:
clinical = (
    data.obs.groupby(["Image"], as_index=False)
    .first()  # first row in each group
    [["Image", "Age", "Grade", "PT", "TMA", "Sex"]]  # sector area columns
)
clinical = clinical.loc[:, ~clinical.columns.duplicated()]

# Remove any non-numeric characters (keep digits and dot)
clinical["PT"] = clinical["PT"].astype(str).str.replace(r'[^0-9.]', '', regex=True)

# Convert to float
clinical["PT"] = pd.to_numeric(clinical["PT"], errors='coerce')
clinical = clinical.set_index('Image')

/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/22639277.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data.obs.groupby(["Image"], as_index=False)


# Distances

In [3]:
cols = ['Image', 'Sector', 'Subtype'] + [col for col in data.obs.columns if col.startswith('Dist_to')] 
distances = pd.DataFrame(data.obs[cols])

counts = (distances.groupby(["Image", "Subtype", "Sector"]).size())
valid_groups = counts[counts > 10].index
distances = distances.groupby(['Image', 'Subtype', 'Sector']).mean().loc[valid_groups].reset_index()

# Melt the marker columns to long format
distances_long = distances.melt(
    id_vars=['Image', 'Subtype', 'Sector'],
)
distances_long['Feature'] = distances_long['Subtype'].astype(str) + "_" + distances_long['Sector'].astype(str) + "_" + distances_long['variable'].astype(str)

# Pivot so that Images are rows and Features are columns
distances_wide = distances_long.pivot_table(
    index='Image',
    columns='Feature',
    values='value'
)
nonzero_var_cols = distances_wide.columns[distances_wide.var() > 0]
distances_wide = distances_wide[nonzero_var_cols]

/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_1803/3230757940.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = (distances.groupby(["Image", "Subtype", "Sector"]).size())
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_1803/3230757940.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  distances = distances.groupby(['Image', 'Subtype', 'Sector']).mean().loc[valid_groups].reset_index()
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_1803/3230757940.py:15: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future v

# Neighbors

In [52]:
cols = ['Image', 'Sector', 'Subtype'] + [col for col in data.obs.columns if col.startswith('Neighboring')] 
neighbors = pd.DataFrame(data.obs[cols])

counts = (neighbors.groupby(["Image", "Subtype", "Sector"]).size())
valid_groups = counts[counts > 10].index
neighbors = neighbors.groupby(['Image', 'Subtype', 'Sector']).mean().loc[valid_groups].reset_index()

# Melt the marker columns to long format
neighbors_long = neighbors.melt(
    id_vars=['Image', 'Subtype', 'Sector'],
)
neighbors_long['Feature'] = neighbors_long['Subtype'].astype(str) + "_" + neighbors_long['Sector'].astype(str) + "_" + neighbors_long['variable'].astype(str)

# Pivot so that Images are rows and Features are columns
neighbors_wide = neighbors_long.pivot_table(
    index='Image',
    columns='Feature',
    values='value'
)
nonzero_var_cols = neighbors_wide.columns[neighbors_wide.var() > 0]
neighbors_wide = neighbors_wide[nonzero_var_cols]


/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/3890008162.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  counts = (neighbors.groupby(["Image", "Subtype", "Sector"]).size())
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/3890008162.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  neighbors = neighbors.groupby(['Image', 'Subtype', 'Sector']).mean().loc[valid_groups].reset_index()
/var/folders/yw/jrjtbzm57tbfg8vxk7g41v9h0000gp/T/ipykernel_2098/3890008162.py:15: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future v

# Outcome

In [ ]:
outcome_df  = data.obs[['Image','Casenumber', 'Recurrence_3yrs', 'Recurrence_5yrs', 'PFS']].drop_duplicates().set_index('Image')
outcome_df.sort_index(inplace=True)

# Export

In [ ]:
ratios_wide.to_csv(os.path.join(base_dir, "Stabl", "dataframes", "celltype_ratios.csv"), index=True)
sector_ratio_df.to_csv(os.path.join(base_dir, "Stabl", "dataframes", "sector_celltype_ratios.csv"), index=True)
freq_wide.to_csv(os.path.join(base_dir, "Stabl", "dataframes", "celltype_frequencies.csv"), index=True)
densities_wide.to_csv(os.path.join(base_dir, "Stabl", "dataframes", "celltype_densities.csv"), index=True)
neighbors_wide.to_csv(os.path.join(base_dir, "Stabl", "dataframes", "neighbors.csv"), index=True)
median_wide.to_csv(os.path.join(base_dir, "Stabl", "dataframes", "functional.csv"), index=True)
distances_wide.to_csv(os.path.join(base_dir, "Stabl", "dataframes", "distances.csv"), index=True)
areas_wide.to_csv(os.path.join(base_dir, "Stabl", "dataframes", "areas.csv"), index=True)
clinical.to_csv(os.path.join(base_dir, "Stabl", "dataframes", "clinical.csv"), index=True)
outcome_df.to_csv(os.path.join(base_dir, "Stabl", "dataframes", "outcomes.csv"), index=True)